# IND320 – Project work, part 2: Data sources
**Course:** IND320 Data to decision (NMBU, autumn 2026)
**Student:** Zarar Zafar

## Links
- **GitHub repository:** [https://github.com/zararzafar/zarar](https://github.com/zararzafar/zarar) (work for this part is on the branch `part2`)
- **Streamlit app:** [https://zarar-ind320.streamlit.app](https://zarar-ind320.streamlit.app)

## Contents
1. Work log
2. Setup (packages, tokens, Spark)
3. NVE Magasinstatistikk API (replaces the CSV file)
4. Part 1 plots recreated from the API
5. ENTSO-E cross-border flows
6. Storing the flows in Cassandra with Spark
7. Extracting data with Spark and plotting
8. Inserting the Spark-extracted data into MongoDB

> **Status of this hand-in (10.11.2026):** Sections 1–4 (NVE API) are complete and run.
> Sections 5–8 depend on an ENTSO-E API token that has been requested but not yet approved. Their
> code is included below as plain text and marked *pending*; the notebook will be resubmitted with
> these sections run as soon as the token arrives.

## 1. Work log
*(300–500 words, describing both the Jupyter Notebook and the Streamlit work)*

In this part the project moves from a static CSV file to live data sources and databases. To keep
the reviewed version of part 1 untouched, all new work is on a separate GitHub branch called `part2`,
which will be merged into `main` after peer review and feedback.

The first task was to replace `reservoirs.csv` with the NVE Magasinstatistikk API. The endpoint
`HentOffentligData` returns the same columns as the CSV file, so the English column names from part 1
could be reused. The lesson about time columns from part 1 still applies: the rows are not sorted, so
the data are sorted by area and date before plotting. The endpoint `HentOmråder` lists the areas:
the whole country (`NO`), the five price areas NO1–NO5 (`EL`) and the water-course areas (`VASS`).
Interestingly, `HentOmråder` lists four water-course areas while the data only contain VASS1–VASS3.
The API also contains newer weeks than the CSV file, and some older values have been revised by NVE,
so the API is the better source. The part 1 plots are recreated from the API in section 4.

The second data source is the ENTSO-E Transparency Platform. Access to its API requires a personal
security token, which has been requested but not yet approved. The code for sections 5–8 is therefore
written and included, but marked as pending and not yet run: downloading cross-border physical flows
for all connections between the Norwegian bidding zones and neighbouring zones for the last two
years with `entsoe-py`, resampling 15-minute values to hourly values, writing them to Cassandra with
Spark, extracting the relevant columns with Spark for a pie chart and a line plot, and inserting
daily sums into MongoDB. These sections will be run and the notebook resubmitted as soon as the token
is approved.

In the Streamlit app the CSV file is removed. The reservoir pages now download data from the NVE API,
and the month slider in the sidebar decides which period is downloaded. Radio buttons choose between
`NO`, `EL` and `VASS`, and both the plot and the `LineChartColumn` table show one line per area
number. The sidebar controls are remembered when switching between the table page and the plot page.
A new page for cross-border flows is prepared: it uses `st.columns` with radio buttons, a year
selector and a pie chart on the left, `st.pills` for months with a line plot on the right, and an
expander describing the data sources. It reads from MongoDB through Streamlit secrets, so no password
is stored on GitHub, and it will show data once the ENTSO-E data have been loaded. The old page four
has been moved to page five.

The main challenges so far have been understanding the structure of the NVE API and planning how the
ENTSO-E data should flow through Cassandra, Spark and MongoDB to the app.

## 2. Setup
Packages used (install once with pip if needed):
`pip install requests pandas matplotlib entsoe-py pyspark cassandra-driver pymongo`

Tokens and passwords are **not** written in the notebook. They are read from environment variables
(or a local file that is listed in `.gitignore`):
- `ENTSOE_TOKEN` – security token from the ENTSO-E Transparency Platform
- `MONGO_URI` – connection string for MongoDB Atlas

In [ ]:
# Standard libraries and data handling
import os
import calendar
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Read secrets from environment variables. Alternatively, put them in a local file
# 'secrets_local.py' (listed in .gitignore) with ENTSOE_TOKEN = "..." and MONGO_URI = "...".
try:
    from secrets_local import ENTSOE_TOKEN, MONGO_URI
except ImportError:
    ENTSOE_TOKEN = os.environ.get("ENTSOE_TOKEN")
    MONGO_URI = os.environ.get("MONGO_URI")

print("ENTSO-E token found:", ENTSOE_TOKEN is not None)
print("MongoDB URI found:  ", MONGO_URI is not None)

## 3. NVE Magasinstatistikk API
The API is documented at <https://biapi.nve.no/magasinstatistikk/swagger>. Two endpoints are used:
- `HentOmråder` – the areas covered by the statistics
- `HentOffentligData` – all weekly observations (the same columns as the CSV file in part 1)

In [ ]:
# Base address of the NVE Magasinstatistikk API
NVE_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"

# Which areas does the API cover? The answer has three groups: land, elspot and vassdrag.
areas_json = requests.get(f"{NVE_URL}/HentOmråder", timeout=30).json()[0]
areas = pd.DataFrame([area for group in areas_json.values() for area in group])
areas[["omrType", "omrnr", "navn", "beskrivelse"]]

In [ ]:
# Download all weekly reservoir data from the API
response = requests.get(f"{NVE_URL}/HentOffentligData", timeout=60)
response.raise_for_status()          # stop with an error message if the request failed
raw = pd.DataFrame(response.json())

# Same English column names as in part 1
column_names = {
    "dato_Id": "date", "omrType": "area_type", "omrnr": "area_number",
    "iso_aar": "iso_year", "iso_uke": "iso_week", "fyllingsgrad": "fill_rate",
    "kapasitet_TWh": "capacity_TWh", "fylling_TWh": "stored_energy_TWh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "fill_rate_previous_week",
    "endring_fyllingsgrad": "fill_rate_change",
}
reservoirs = raw.rename(columns=column_names)

# Time columns: 'date' is the date of the weekly observation (a Sunday).
# Lesson from part 1: the rows are not sorted, so sort by area and date.
reservoirs["date"] = pd.to_datetime(reservoirs["date"])
reservoirs = reservoirs.sort_values(["area_type", "area_number", "date"]).reset_index(drop=True)

print("Rows:", len(reservoirs))
print("Period:", reservoirs["date"].min().date(), "to", reservoirs["date"].max().date())
print(reservoirs.groupby(["area_type", "area_number"]).size().rename("weeks").to_string())

## 4. Part 1 plots recreated from the API
As in part 1, the national total for Norway (`area_type == "NO"`) is used.

In [ ]:
# National series for Norway, indexed by date
norway = reservoirs[reservoirs["area_type"] == "NO"].set_index("date")
value_columns = ["fill_rate", "capacity_TWh", "stored_energy_TWh",
                 "fill_rate_previous_week", "fill_rate_change"]
labels = {
    "fill_rate": ("Fill rate", "Share of capacity (0-1)"),
    "capacity_TWh": ("Reservoir capacity", "TWh"),
    "stored_energy_TWh": ("Stored energy", "TWh"),
    "fill_rate_previous_week": ("Fill rate previous week", "Share of capacity (0-1)"),
    "fill_rate_change": ("Weekly change in fill rate", "Change (fraction points)"),
}

# Each column on its own axis
fig, axes = plt.subplots(len(value_columns), 1, figsize=(11, 13), sharex=True)
for ax, col in zip(axes, value_columns):
    ax.plot(norway.index, norway[col], linewidth=0.8)
    ax.set_title(labels[col][0])
    ax.set_ylabel(labels[col][1])
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Date")
fig.suptitle("Norwegian hydropower reservoirs from the NVE API (weekly)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# All columns together: min-max scaling (constant columns are drawn at 0.5)
def min_max_scale(frame):
    value_range = frame.max() - frame.min()
    scaled = (frame - frame.min()) / value_range.replace(0, 1)
    scaled.loc[:, value_range == 0] = 0.5
    return scaled

recent = min_max_scale(norway[value_columns])
recent = recent[recent.index >= recent.index.max() - pd.DateOffset(years=3)]

fig, ax = plt.subplots(figsize=(11, 5))
for col in value_columns:
    ax.plot(recent.index, recent[col], label=labels[col][0], linewidth=1.3)
ax.set_title("All columns together, min-max scaled (last three years, NVE API)")
ax.set_xlabel("Date")
ax.set_ylabel("Scaled value (0 = min, 1 = max)")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 5. ENTSO-E cross-border flows
**⏳ Pending ENTSO-E token** – the code in this section is written but not yet run.

Cross-border *physical flows* (ENTSO-E document type A11) are downloaded with the `entsoe-py`
package for every connection between a Norwegian bidding zone and a neighbouring zone, in both
directions, for the last two years. `entsoe-py` splits long periods into one-year requests itself.

## 6. Storing the flows in Cassandra with Spark
**⏳ Pending ENTSO-E token** – the code in this section is written but not yet run.

The keyspace and table are created with the Cassandra Python driver. The partition key is
(connection, direction) and the rows are ordered by time within each partition.

## 7. Extracting data with Spark and plotting
**⏳ Pending ENTSO-E token** – the code in this section is written but not yet run.

Spark reads the table back from Cassandra and selects only the columns that are needed.

## 8. Inserting the Spark-extracted data into MongoDB
**⏳ Pending ENTSO-E token** – the code in this section is written but not yet run.

Spark sums the hourly values to daily energy (MWh) per connection and direction. These daily
values are inserted into MongoDB (database `ind320`, collection `crossborder_daily`), which the
Streamlit app reads.

## Next steps
In part 3 the work continues on the same branch structure, building on the API connections and the
MongoDB data used by the Streamlit app.